# Script 2a — Spectral distance measurements

This notebook compares individual phee calls with the same three measurements used in the original analysis:

- **DTW:** FastDTW alignment cost between 4–12 kHz spectrograms.
- **Traditional acoustics:** Euclidean distance between five traditional-measure PCA scores.
- **MFCC:** Euclidean distance between five MFCC PCA scores.

Computing every DTW pair is expensive. The normal workflow loads saved distance matrices from `distance_matrices/`; it does not reopen the WAV files. The recomputation cell is retained for completeness but is disabled by default.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# Safe defaults: use the matrices produced during the original analysis.
RECOMPUTE_CALL_DISTANCES = False
DISTANCE_TAG = "2200calls"

# These settings matter only when RECOMPUTE_CALL_DISTANCES is set to True.
RECOMPUTE_SAMPLE_SIZE = 2200
N_JOBS = -1

working_directory = Path.cwd().resolve()
BASE_DIR = working_directory.parent if working_directory.name == "code" else working_directory
SPECTRAL_DATA_DIR = BASE_DIR / "spectral data"
DISTANCE_MATRIX_DIR = BASE_DIR / "distance_matrices"
EXTRACTED_CALL_DIR = BASE_DIR / "extracted calls"

print("Repository:", BASE_DIR)
print("Recompute call distances:", RECOMPUTE_CALL_DISTANCES)

## 1. Load the saved matrices

All four files must use the same tag because the row order in the label table defines the rows and columns of every matrix. Change `DISTANCE_TAG` above if a different saved subset is being used.

In [ ]:
cache_paths = {
    "DTW": DISTANCE_MATRIX_DIR / f"dtw_dist_matrix_{DISTANCE_TAG}.npy",
    "Traditional acoustics": DISTANCE_MATRIX_DIR
    / f"trad_pca_dist_matrix_{DISTANCE_TAG}.npy",
    "MFCC": DISTANCE_MATRIX_DIR / f"mfcc_pca_dist_matrix_{DISTANCE_TAG}.npy",
}
label_path = DISTANCE_MATRIX_DIR / f"labels_{DISTANCE_TAG}.csv"
missing_cache = [path for path in [*cache_paths.values(), label_path] if not path.exists()]

distance_matrices = {}
call_labels = pd.DataFrame()
cache_ready = not missing_cache

if not cache_ready:
    print(
        f"Saved matrices for tag {DISTANCE_TAG!r} were not found. "
        "Place the three .npy files and matching labels CSV in distance_matrices/, "
        "then rerun this notebook. Recalculation remains off."
    )
else:
    distance_matrices = {
        name: np.load(path) for name, path in cache_paths.items()
    }
    call_labels = pd.read_csv(label_path)
    expected_shape = (len(call_labels), len(call_labels))

    for name, matrix in distance_matrices.items():
        if matrix.shape != expected_shape:
            raise ValueError(
                f"{name} has shape {matrix.shape}; expected {expected_shape} from {label_path.name}."
            )
        if not np.allclose(matrix, matrix.T, equal_nan=True):
            raise ValueError(f"{name} is not symmetric.")

    print(f"Loaded {len(call_labels):,} calls from cache tag {DISTANCE_TAG!r}.")
    display(call_labels[["filename", "stage", "focal ID", "conspecific_ID"]].head())

## 2. Optional: recompute all call distances

This is the single expensive branch in the notebook. When enabled, it:

1. resolves the extracted WAV paths in `Processed_spec_data.csv`;
2. computes Hann-window STFTs with a 2,048-sample window and 512-sample hop;
3. keeps frequencies from 4 to 12 kHz;
4. evaluates every upper-triangle FastDTW pair in parallel with cosine frame distance; and
5. saves the DTW, traditional-PCA, and MFCC-PCA matrices with a matching label table.

For the released workflow, leave `RECOMPUTE_CALL_DISTANCES = False` and supply the saved matrices instead.

In [ ]:
if RECOMPUTE_CALL_DISTANCES:
    import os
    from contextlib import contextmanager

    import joblib
    import librosa
    from fastdtw import fastdtw
    from joblib import Parallel, delayed
    from scipy.spatial.distance import cosine, pdist, squareform
    from tqdm.auto import tqdm


    @contextmanager
    def tqdm_joblib(progress_bar):
        # Update a tqdm progress bar after each completed joblib batch.
        original_callback = joblib.parallel.BatchCompletionCallBack

        class ProgressCallback(original_callback):
            def __call__(self, *args, **kwargs):
                progress_bar.update(n=self.batch_size)
                return super().__call__(*args, **kwargs)

        joblib.parallel.BatchCompletionCallBack = ProgressCallback
        try:
            yield progress_bar
        finally:
            joblib.parallel.BatchCompletionCallBack = original_callback
            progress_bar.close()


    def resolve_audio_path(stored_path):
        # Use the repository copy even when the CSV contains an older absolute path.
        filename = Path(str(stored_path).replace("\\", "/")).name
        return EXTRACTED_CALL_DIR / filename


    def load_spectrogram_band(audio_path):
        # Read one call and return its 4–12 kHz dB spectrogram.
        samples, sample_rate = librosa.load(audio_path, sr=None, mono=True)
        spectrum = librosa.stft(
            samples,
            n_fft=2048,
            hop_length=512,
            win_length=2048,
            window="hann",
        )
        spectrogram_db = librosa.amplitude_to_db(np.abs(spectrum), ref=np.max)
        frequencies = librosa.fft_frequencies(sr=sample_rate, n_fft=2048)
        return spectrogram_db[(frequencies >= 4_000) & (frequencies <= 12_000)]


    def fastdtw_pair(index_a, index_b, path_a, path_b):
        # Calculate the normalized alignment cost for one pair of calls.
        spectrogram_a = load_spectrogram_band(path_a)
        spectrogram_b = load_spectrogram_band(path_b)
        distance, _ = fastdtw(
            spectrogram_a.T,
            spectrogram_b.T,
            dist=cosine,
        )
        normalized_distance = distance / (
            spectrogram_a.shape[1] + spectrogram_b.shape[1]
        )
        return index_a, index_b, normalized_distance


    processed_calls = pd.read_csv(SPECTRAL_DATA_DIR / "Processed_spec_data.csv")
    processed_calls["_audio_file"] = processed_calls["call_audio_path"].map(
        resolve_audio_path
    )
    available_calls = processed_calls[
        processed_calls["_audio_file"].map(Path.exists)
    ].reset_index(drop=True)
    if available_calls.empty:
        raise FileNotFoundError(
            "No extracted call WAV files match the paths in Processed_spec_data.csv."
        )

    if RECOMPUTE_SAMPLE_SIZE and RECOMPUTE_SAMPLE_SIZE < len(available_calls):
        subset = available_calls.sample(
            RECOMPUTE_SAMPLE_SIZE, random_state=42
        ).reset_index(drop=True)
    else:
        subset = available_calls.copy()

    number_of_calls = len(subset)
    jobs = [
        (i, j, subset.loc[i, "_audio_file"], subset.loc[j, "_audio_file"])
        for i in range(number_of_calls)
        for j in range(i + 1, number_of_calls)
    ]
    print(
        f"Computing {len(jobs):,} DTW pairs for {number_of_calls:,} calls "
        f"with {os.cpu_count() if N_JOBS == -1 else N_JOBS} worker(s)."
    )

    if N_JOBS == 1:
        pair_results = [
            fastdtw_pair(*job) for job in tqdm(jobs, desc="DTW pairs")
        ]
    else:
        with tqdm_joblib(tqdm(total=len(jobs), desc="DTW pairs")):
            pair_results = Parallel(n_jobs=N_JOBS, backend="loky")(
                delayed(fastdtw_pair)(*job) for job in jobs
            )

    dtw_matrix = np.zeros((number_of_calls, number_of_calls), dtype=float)
    for index_a, index_b, distance in pair_results:
        dtw_matrix[index_a, index_b] = distance
        dtw_matrix[index_b, index_a] = distance

    traditional_columns = [f"trad_PC{component}" for component in range(1, 6)]
    mfcc_columns = [f"mfcc_PC{component}" for component in range(1, 6)]
    traditional_matrix = squareform(pdist(subset[traditional_columns], "euclidean"))
    mfcc_matrix = squareform(pdist(subset[mfcc_columns], "euclidean"))

    DISTANCE_MATRIX_DIR.mkdir(parents=True, exist_ok=True)
    DISTANCE_TAG = f"{number_of_calls}calls"
    np.save(
        DISTANCE_MATRIX_DIR / f"dtw_dist_matrix_{DISTANCE_TAG}.npy",
        dtw_matrix,
    )
    np.save(
        DISTANCE_MATRIX_DIR / f"trad_pca_dist_matrix_{DISTANCE_TAG}.npy",
        traditional_matrix,
    )
    np.save(
        DISTANCE_MATRIX_DIR / f"mfcc_pca_dist_matrix_{DISTANCE_TAG}.npy",
        mfcc_matrix,
    )
    subset.drop(columns="_audio_file").to_csv(
        DISTANCE_MATRIX_DIR / f"labels_{DISTANCE_TAG}.csv",
        index=False,
    )

    distance_matrices = {
        "DTW": dtw_matrix,
        "Traditional acoustics": traditional_matrix,
        "MFCC": mfcc_matrix,
    }
    call_labels = subset.drop(columns="_audio_file").copy()
    cache_ready = True
    print(f"Saved distance matrices with tag {DISTANCE_TAG!r}.")

## 3. Compare the three distance measures

The matrices are symmetric, so each pair should be counted once. Pearson correlations are therefore calculated from the upper triangle, excluding the diagonal.

In [ ]:
if cache_ready:
    from itertools import combinations

    from scipy.stats import pearsonr


    def upper_triangle(matrix):
        return matrix[np.triu_indices_from(matrix, k=1)]


    correlation_rows = []
    for name_a, name_b in combinations(distance_matrices, 2):
        correlation, p_value = pearsonr(
            upper_triangle(distance_matrices[name_a]),
            upper_triangle(distance_matrices[name_b]),
        )
        correlation_rows.append(
            {
                "measure_a": name_a,
                "measure_b": name_b,
                "pearson_r": correlation,
                "p_value": p_value,
            }
        )

    distance_correlations = pd.DataFrame(correlation_rows)
    display(distance_correlations.round({"pearson_r": 3}))
else:
    print("Distance correlations will be shown after the saved matrices are added.")

## 4. Visualize calls with UMAP

Each distance matrix is projected once with UMAP. Panels separate the before and after stages, points are coloured by the focal individual, and larger symbols mark each individual's centroid within partner and non-partner contexts. This is an exploratory view of the call-level structure; the later modelling scripts operate on the distance values themselves.

In [ ]:
if cache_ready:
    import matplotlib.pyplot as plt
    import umap
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    partner_map = {
        "Tabor": "Lola", "Lola": "Tabor",
        "Odin": "Nougatti", "Nougatti": "Odin",
        "Wuschel": "Olympia", "Olympia": "Wuschel",
    }
    call_labels["partner_status"] = np.where(
        call_labels["focal ID"].map(partner_map).eq(call_labels["conspecific_ID"]),
        "partner",
        "non-partner",
    )

    embeddings = {}
    for measure, matrix in distance_matrices.items():
        coordinates = umap.UMAP(
            metric="precomputed",
            n_neighbors=30,
            min_dist=0.1,
            random_state=2,
        ).fit_transform(matrix)
        embeddings[measure] = call_labels.assign(
            UMAP1=coordinates[:, 0],
            UMAP2=coordinates[:, 1],
        )

    palette = {
        "Tabor": "#2e1571",
        "Odin": "#df2e12",
        "Wuschel": "#6D9290",
        "Lola": "#4725DD",
        "Nougatti": "#ffd700",
        "Olympia": "#ffffff",
    }
    point_markers = {"partner": "X", "non-partner": "o"}
    centroid_markers = {"partner": "^", "non-partner": "s"}
    stages = ["before", "after"]

    figure, axes = plt.subplots(
        len(distance_matrices),
        len(stages),
        figsize=(14, 17),
        squeeze=False,
    )

    for row_index, (measure, embedded_calls) in enumerate(embeddings.items()):
        for column_index, stage in enumerate(stages):
            axis = axes[row_index, column_index]
            stage_calls = embedded_calls[embedded_calls["stage"].eq(stage)]

            for status, marker in point_markers.items():
                points = stage_calls[stage_calls["partner_status"].eq(status)]
                axis.scatter(
                    points["UMAP1"],
                    points["UMAP2"],
                    c=points["focal ID"].map(palette),
                    marker=marker,
                    edgecolor="black",
                    linewidth=0.35,
                    s=35,
                    alpha=0.5,
                )

            centroids = (
                stage_calls.groupby(["focal ID", "partner_status"], observed=True)
                [["UMAP1", "UMAP2"]]
                .mean()
                .reset_index()
            )
            for _, centroid in centroids.iterrows():
                axis.scatter(
                    centroid["UMAP1"],
                    centroid["UMAP2"],
                    marker=centroid_markers[centroid["partner_status"]],
                    s=220,
                    facecolor=palette[centroid["focal ID"]],
                    edgecolor="black",
                    linewidth=1.2,
                    zorder=5,
                )

            axis.set_title(f"{measure} — {stage.capitalize()} (n={len(stage_calls):,})")
            axis.set_xlabel("UMAP 1")
            axis.set_ylabel("UMAP 2")

    individual_handles = [
        Patch(facecolor=colour, edgecolor="black", label=individual)
        for individual, colour in palette.items()
    ]
    symbol_handles = [
        Line2D(
            [], [], marker=marker, linestyle="", color="grey",
            markeredgecolor="black", label=status,
        )
        for status, marker in point_markers.items()
    ] + [
        Line2D(
            [], [], marker=marker, linestyle="", color="grey",
            markeredgecolor="black", markersize=10, label=f"{status} centroid",
        )
        for status, marker in centroid_markers.items()
    ]
    figure.legend(
        handles=individual_handles + symbol_handles,
        loc="upper center",
        ncol=5,
        bbox_to_anchor=(0.5, 1.0),
        frameon=False,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()
else:
    print("The UMAP figure will be created after the saved matrices are added.")